# Rock energy segmentation with hardness_score_smooth window 30

Изменения относительно текущей версии:

```text
hardness_score_smooth = zscore(log_pseudo_mse_roll_median_30)
```

То есть сам признак энергоёмкости теперь строится по rolling-окну 30, а не 60.

`SEGMENT_SIZE = 60` оставлен без изменения: это размер блока для агрегации сегментов и расчёта квантильных классов, а не окно сглаживания `hardness_score_smooth`.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.ndimage import gaussian_filter

EPS = 1e-6

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [2]:
DATA_PATH = "../datasets/united.csv"

_df = pd.read_csv(DATA_PATH)
_df = _df.drop(columns=["Unnamed: 0"], errors="ignore")

if "depth_m" not in _df.columns:
    if "depth" not in _df.columns:
        raise ValueError("Missing required depth/depth_m column in united.csv")
    _df["depth_m"] = _df["depth"]

_df["depth_m"] = pd.to_numeric(_df["depth_m"], errors="coerce")
df = _df

required_cols = [
    "processing_time",
    "well_id",
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "speed",
    "depth_m",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"?? ??????? ???????: {missing}")

df = df.dropna(subset=["depth_m"]).copy()
df["processing_time"] = pd.to_datetime(df["processing_time"])
df = df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
print("Has depth:", "depth" in df.columns)
print("Has depth_m:", "depth_m" in df.columns)
display(df[required_cols].head())
display(df[["pressure_axis", "pressure_rotation", "rotation", "speed", "depth_m"]].describe(percentiles=[.01, .05, .5, .95, .99]))

Loaded: ../datasets/united.csv
Shape: (415049, 8)
Has depth: True
Has depth_m: True


,processing_time,well_id,pressure_axis,pressure_rotation,rotation,speed,depth_m
0,2025-08-24 10:09:50.980,19601,804,4551,74.256,0.002755,0.0606
1,2025-08-24 10:10:00.260,19601,763,3782,73.812,0.003030,0.0909
2,2025-08-24 10:10:05.199,19601,879,4407,73.512,0.006060,0.1212
3,2025-08-24 10:10:14.610,19601,721,3705,73.962,0.002755,0.1515
4,2025-08-24 10:10:34.197,19601,859,3883,74.256,0.001515,0.1818


,pressure_axis,pressure_rotation,rotation,speed,depth_m
count,415049.000000,415049.000000,415049.000000,415049.000000,415049.000000
mean,17473.667273,14134.146325,103.945315,0.013116,10.733979
std,4682.982816,3243.523440,13.370361,0.006615,7.873565
min,317.000000,784.000000,50.010000,0.001002,-0.848400
1%,3713.000000,6271.000000,64.980000,0.002755,0.363600
5%,6645.000000,8246.000000,81.750000,0.005050,1.212000
50%,18861.000000,14637.000000,103.158000,0.012120,9.696000
95%,22343.000000,18758.600000,138.474000,0.024240,23.482500
99%,23626.000000,20881.000000,139.020000,0.030300,40.511100
max,24872.000000,26318.000000,139.578000,0.038957,82.052400


In [3]:
df["dt"] = (df.groupby("well_id")["processing_time"].diff().dt.total_seconds())
df["dt"] = df["dt"].fillna(df["dt"].median())

df["total_pressure"] = df["pressure_axis"] + df["pressure_rotation"]
df["pressure_balance"] = df["pressure_axis"] / (df["total_pressure"] + EPS)
df["axis_over_rot_pressure"] = df["pressure_axis"] / (df["pressure_rotation"] + EPS)
df["rot_pressure_over_axis"] = df["pressure_rotation"] / (df["pressure_axis"] + EPS)
df["rotation_efficiency"] = df["rotation"] / (df["pressure_rotation"] + EPS)

df["axis_x_rotation"] = df["pressure_axis"] * df["rotation"]
df["rot_pressure_x_rotation"] = df["pressure_rotation"] * df["rotation"]

# Proxy подводимого воздействия и proxy-MSE.
# Чем больше pseudo_mse, тем больше условной энергии требуется на единицу проходки.
df["energy_input_proxy"] = df["pressure_axis"] + df["pressure_rotation"] * df["rotation"]
df["pseudo_mse"] = df["energy_input_proxy"] / (df["speed"] + EPS)

df["log_energy_input_proxy"] = np.log1p(df["energy_input_proxy"])
df["log_pseudo_mse"] = np.log1p(df["pseudo_mse"])

display(df[[
    "dt", "energy_input_proxy", "pseudo_mse", "log_pseudo_mse",
    "rotation_efficiency", "pressure_balance",
]].describe(percentiles=[.01, .05, .5, .95, .99]))

,dt,energy_input_proxy,pseudo_mse,log_pseudo_mse,rotation_efficiency,pressure_balance
count,415049.000000,4.150490e+05,4.150490e+05,415049.000000,415049.000000,415049.000000
mean,7.270602,1.494500e+06,1.467727e+08,18.649861,0.007803,0.545942
std,110.393457,4.009753e+05,9.155477e+07,0.552515,0.002483,0.057506
min,0.089000,4.174649e+04,3.009300e+06,14.917218,0.002104,0.013293
1%,0.229000,4.697380e+05,3.257078e+07,17.298926,0.004789,0.320485
5%,0.464000,7.635049e+05,5.221720e+07,17.770923,0.005407,0.428681
50%,5.135000,1.542361e+06,1.239899e+08,18.635711,0.007111,0.558077
95%,10.891000,2.119089e+06,3.036335e+08,19.531332,0.012153,0.607300
99%,23.689560,2.462635e+06,4.641533e+08,19.955725,0.015835,0.629086
max,42583.603000,3.668551e+06,2.581497e+09,21.671635,0.121481,0.933514


In [4]:
def add_rolling_stats(data, cols, windows=(12, 30, 60), group_col="well_id"):
    out = data.copy()

    for col in cols:
        for w in windows:
            min_p = max(3, w // 3)

            out[f"{col}_roll_median_{w}"] = (
                out.groupby(group_col)[col]
                   .transform(lambda s: s.rolling(w, min_periods=min_p).median())
            )

            out[f"{col}_roll_mean_{w}"] = (
                out.groupby(group_col)[col]
                   .transform(lambda s: s.rolling(w, min_periods=min_p).mean())
            )

            out[f"{col}_roll_std_{w}"] = (
                out.groupby(group_col)[col]
                   .transform(lambda s: s.rolling(w, min_periods=min_p).std())
            )

    return out

rolling_cols = [
    "energy_input_proxy",
    "pseudo_mse",
    "log_pseudo_mse",
    "rotation_efficiency",
    "speed",
    "rotation",
    "pressure_balance",
]

df = add_rolling_stats(df, rolling_cols, windows=(12, 30, 60))

print("Rolling features added:", len([c for c in df.columns if "_roll_" in c]))

Rolling features added: 63


In [5]:
def zscore(s):
    std = s.std()
    if not np.isfinite(std) or std < EPS:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / (std + EPS)

# Основной непрерывный признак энергоёмкости.
# Он основан только на сглаженном log-pseudo-MSE по окну 30.
df["hardness_score_smooth"] = zscore(df["log_pseudo_mse_roll_median_30"])

HARDNESS_FEATURES = ["hardness_score_smooth"]

print("Hardness features:")
print(HARDNESS_FEATURES)

display(df[HARDNESS_FEATURES + [
    "log_pseudo_mse",
    "log_pseudo_mse_roll_median_30",
]].describe(percentiles=[.01, .05, .5, .95, .99]))

# Diagnostics for new 30-window hardness signal.
print("hardness_score_smooth NaN count:", int(df["hardness_score_smooth"].isna().sum()))
print("hardness_score_smooth NaN pct:", float(df["hardness_score_smooth"].isna().mean() * 100))


Hardness features:
['hardness_score_smooth']


,hardness_score_smooth,log_pseudo_mse,log_pseudo_mse_roll_median_30
count,3.996950e+05,415049.000000,399695.000000
mean,-3.121947e-15,18.649861,18.613582
std,9.999976e-01,0.552515,0.417546
min,-5.885674e+00,14.917218,16.156040
1%,-2.683368e+00,17.298926,17.493151
5%,-1.726845e+00,17.770923,17.892544
50%,8.842769e-02,18.635711,18.650505
95%,1.625172e+00,19.531332,19.292167
99%,2.066702e+00,19.955725,19.476526
max,3.713749e+00,21.671635,20.164245


hardness_score_smooth NaN count: 15354
hardness_score_smooth NaN pct: 3.699322248698347


In [6]:
energy_labels_4 = [
    "soft_low_energy",
    "medium_low_energy",
    "medium_high_energy",
    "hard_high_energy",
]

SEGMENT_SIZE = 60

segment_df = df.dropna(subset=[
    "hardness_score_smooth",
    "pseudo_mse_roll_median_30",
    "speed",
]).copy()

segment_df = segment_df.sort_values(["well_id", "processing_time"]).copy()
segment_df["_row_in_well"] = segment_df.groupby("well_id").cumcount()
segment_df["segment_id"] = (segment_df["_row_in_well"] // SEGMENT_SIZE).astype(int)

segments = (
    segment_df
    .groupby(["well_id", "segment_id"])
    .agg(
        segment_start=("processing_time", "min"),
        segment_end=("processing_time", "max"),
        rows=("speed", "size"),
        hardness_segment=("hardness_score_smooth", "median"),
        pseudo_mse_segment=("pseudo_mse_roll_median_30", "median"),
        speed_segment=("speed", "median"),
    )
    .reset_index()
)

segments["energy_type_segment_quantile"] = pd.qcut(
    segments["hardness_segment"],
    q=4,
    labels=energy_labels_4,
    duplicates="drop",
).astype(str)

segment_df = segment_df.merge(
    segments[["well_id", "segment_id", "hardness_segment", "energy_type_segment_quantile"]],
    on=["well_id", "segment_id"],
    how="left",
)

display(
    segments
    .groupby("energy_type_segment_quantile")
    .agg(
        segments=("segment_id", "size"),
        rows=("rows", "sum"),
        hardness_segment=("hardness_segment", "median"),
        pseudo_mse_segment=("pseudo_mse_segment", "median"),
        speed_segment=("speed_segment", "median"),
    )
    .sort_values("hardness_segment")
)

,segments,rows,hardness_segment,pseudo_mse_segment,speed_segment
energy_type_segment_quantile,,,,,
soft_low_energy,1876,100504,-1.024404,7.912285e+07,0.018180
medium_low_energy,1876,100086,-0.212240,1.110417e+08,0.012966
medium_high_energy,1875,98880,0.297587,1.373776e+08,0.012120
hard_high_energy,1876,100225,1.082099,1.908794e+08,0.006060


In [7]:
merge_cols = [
    "processing_time",
    "well_id",
    "segment_id",
    "hardness_segment",
    "energy_type_segment_quantile",
]

df_out = df.merge(
    segment_df[merge_cols],
    on=["processing_time", "well_id"],
    how="left",
    suffixes=("", "_segment"),
)

if "depth_m" not in df_out.columns:
    if "depth" not in df_out.columns:
        raise ValueError("df_out lost depth/depth_m before save")
    df_out["depth_m"] = df_out["depth"]
df_out["depth_m"] = pd.to_numeric(df_out["depth_m"], errors="coerce")

df_out["rock_energy_type_final"] = df_out["energy_type_segment_quantile"]

display(df_out[[
    "processing_time",
    "well_id",
    "depth_m",
    "speed",
    "energy_input_proxy",
    "pseudo_mse",
    "hardness_score_smooth",
    "segment_id",
    "hardness_segment",
    "rock_energy_type_final",
]].head())

,processing_time,well_id,depth_m,speed,energy_input_proxy,pseudo_mse,hardness_score_smooth,segment_id,hardness_segment,rock_energy_type_final
0,2025-08-24 10:09:50.980,19601,0.0606,0.002755,338743.056,1.229314e+08,NaN,NaN,NaN,NaN
1,2025-08-24 10:10:00.260,19601,0.0909,0.003030,279919.984,9.235235e+07,NaN,NaN,NaN,NaN
2,2025-08-24 10:10:05.199,19601,0.1212,0.006060,324846.384,5.359617e+07,NaN,NaN,NaN,NaN
3,2025-08-24 10:10:14.610,19601,0.1515,0.002755,274750.210,9.970810e+07,NaN,NaN,NaN,NaN
4,2025-08-24 10:10:34.197,19601,0.1818,0.001515,289195.048,1.907619e+08,NaN,NaN,NaN,NaN


In [8]:
summary_final = (
    df_out
    .groupby("rock_energy_type_final")
    .agg(
        rows=("speed", "size"),
        speed_median=("speed", "median"),
        pseudo_mse_smooth=("pseudo_mse_roll_median_30", "median"),
        hardness_smooth=("hardness_score_smooth", "median"),
        pressure_axis_median=("pressure_axis", "median"),
        pressure_rotation_median=("pressure_rotation", "median"),
        rotation_median=("rotation", "median"),
    )
    .sort_values("hardness_smooth")
)

display(summary_final)

,rows,speed_median,pseudo_mse_smooth,hardness_smooth,pressure_axis_median,pressure_rotation_median,rotation_median
rock_energy_type_final,,,,,,,
soft_low_energy,100504,0.01818,7.826618e+07,-1.049528,16396.0,13275.0,103.410
medium_low_energy,100086,0.01212,1.108857e+08,-0.215161,19215.0,15170.0,103.008
medium_high_energy,98880,0.01212,1.373713e+08,0.298085,20396.0,15559.0,102.864
hard_high_energy,100225,0.00606,1.924607e+08,1.100641,19537.0,14351.0,103.458


In [9]:
SURFACE_HTML_DIR = Path("plotly_surfaces_html")
SURFACE_HTML_DIR.mkdir(exist_ok=True)

SURFACE_REPORT_DIR = Path("rock_energy_segment_reports")
SURFACE_REPORT_DIR.mkdir(exist_ok=True)
SURFACE_REPORT_PATH = SURFACE_REPORT_DIR / "surface_empirical_report.csv"

SURFACE_GRID_SIZE = 40
SURFACE_MIN_BIN_COUNT = 5
SURFACE_SMOOTH_SIGMA = 1.6
SURFACE_Z_CLIP_QUANTILES = (0.02, 0.98)
SURFACE_AXIS_QUANTILES = (0.05, 0.95)
SURFACE_TARGET_CANDIDATES = [
    "speed_roll_median_30",
    "speed_roll_mean_30",
    "speed_roll_median_12",
    "speed",
]


def pick_surface_target(data: pd.DataFrame) -> str:
    for col in SURFACE_TARGET_CANDIDATES:
        if col in data.columns and data[col].notna().sum() >= 500:
            return col
    return "speed"


def fill_and_smooth_surface(
    z: np.ndarray,
    counts: np.ndarray,
    min_bin_count: int = SURFACE_MIN_BIN_COUNT,
    sigma: float = SURFACE_SMOOTH_SIGMA,
    clip_quantiles: tuple[float, float] = SURFACE_Z_CLIP_QUANTILES,
) -> tuple[np.ndarray, dict]:
    z = z.astype(float, copy=True)
    counts = counts.astype(float, copy=True)

    support_mask = counts >= min_bin_count
    z[~support_mask] = np.nan
    raw_filled_ratio = float(np.isfinite(z).mean())

    z_df = pd.DataFrame(z)
    z_df = z_df.interpolate(axis=0, limit_direction="both")
    z_df = z_df.interpolate(axis=1, limit_direction="both")
    z_filled = z_df.to_numpy(dtype=float)

    finite_median = np.nanmedian(z_filled)
    if not np.isfinite(finite_median):
        finite_median = 0.0
    z_filled = np.where(np.isfinite(z_filled), z_filled, finite_median)

    weights = np.where(support_mask, np.clip(counts / max(min_bin_count, 1), 0.0, 1.0), 0.0)
    weighted_z = gaussian_filter(z_filled * weights, sigma=sigma)
    weighted_w = gaussian_filter(weights, sigma=sigma)

    z_smooth = np.where(weighted_w > 1e-9, weighted_z / weighted_w, z_filled)
    z_smooth = np.where(np.isfinite(z_smooth), z_smooth, z_filled)

    q_low, q_high = np.nanquantile(z_smooth, clip_quantiles)
    z_clipped = np.clip(z_smooth, q_low, q_high)

    report = {
        "raw_filled_cell_ratio_after_min_count": raw_filled_ratio,
        "z_clip_low": float(q_low),
        "z_clip_high": float(q_high),
    }

    return z_clipped, report


def build_empirical_surface(
    data: pd.DataFrame,
    x_col: str = "pressure_axis",
    y_col: str = "pressure_rotation",
    grid_size: int = SURFACE_GRID_SIZE,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict]:
    z_col = pick_surface_target(data)
    part = data.dropna(subset=[x_col, y_col, z_col]).copy()

    if len(part) < 500:
        raise ValueError(f"Not enough rows for empirical surface: {len(part)}")

    q_low, q_high = SURFACE_AXIS_QUANTILES
    x_min = float(part[x_col].quantile(q_low))
    x_max = float(part[x_col].quantile(q_high))
    y_min = float(part[y_col].quantile(q_low))
    y_max = float(part[y_col].quantile(q_high))

    part = part[
        part[x_col].between(x_min, x_max)
        & part[y_col].between(y_min, y_max)
    ].copy()

    x_edges = np.linspace(x_min, x_max, grid_size + 1)
    y_edges = np.linspace(y_min, y_max, grid_size + 1)

    part["x_bin"] = pd.cut(part[x_col], bins=x_edges, labels=False, include_lowest=True)
    part["y_bin"] = pd.cut(part[y_col], bins=y_edges, labels=False, include_lowest=True)

    grouped = (
        part.dropna(subset=["x_bin", "y_bin"])
        .groupby(["x_bin", "y_bin"], observed=True)[z_col]
        .agg(["median", "count"])
    )

    z = np.full((grid_size, grid_size), np.nan, dtype=float)
    counts = np.zeros((grid_size, grid_size), dtype=float)

    for (x_bin, y_bin), row in grouped.iterrows():
        xi = int(x_bin)
        yi = int(y_bin)
        if 0 <= xi < grid_size and 0 <= yi < grid_size:
            z[xi, yi] = float(row["median"])
            counts[xi, yi] = float(row["count"])

    filled_ratio_before_min_count = float(np.isfinite(z).mean())
    z_smooth, smooth_report = fill_and_smooth_surface(z, counts)

    x_centers = (x_edges[:-1] + x_edges[1:]) / 2.0
    y_centers = (y_edges[:-1] + y_edges[1:]) / 2.0
    pa, pr = np.meshgrid(x_centers, y_centers, indexing="ij")

    positive_counts = counts[counts > 0]
    report = {
        "surface_target": z_col,
        "rows_used": int(len(part)),
        "grid_size": int(grid_size),
        "min_bin_count": int(SURFACE_MIN_BIN_COUNT),
        "smooth_sigma": float(SURFACE_SMOOTH_SIGMA),
        "filled_cell_ratio_before_min_count": filled_ratio_before_min_count,
        "nonzero_bin_count": int((counts > 0).sum()),
        "median_bin_count": float(np.median(positive_counts)) if len(positive_counts) else 0.0,
        "pressure_axis_q_low": x_min,
        "pressure_axis_q_high": x_max,
        "pressure_rotation_q_low": y_min,
        "pressure_rotation_q_high": y_max,
        "surface_speed_min": float(np.nanmin(z_smooth)),
        "surface_speed_median": float(np.nanmedian(z_smooth)),
        "surface_speed_max": float(np.nanmax(z_smooth)),
        **smooth_report,
    }

    return pa, pr, z_smooth, report


surface_reports = []

for surface_type in energy_labels_4:
    surface_train = df_out[df_out["rock_energy_type_final"] == surface_type].copy()

    if len(surface_train) < 500:
        print("Skip small class:", surface_type, len(surface_train))
        continue

    pa, pr, z, report = build_empirical_surface(surface_train)
    report = {"rock_energy_type_final": surface_type, **report}
    surface_reports.append(report)

    fig = go.Figure()
    fig.add_trace(go.Surface(
        x=pa,
        y=pr,
        z=z,
        colorscale="Viridis",
        opacity=0.95,
        showscale=True,
    ))

    fig.update_layout(
        title=f"Smoothed empirical surface by energy type: {surface_type}",
        scene=dict(
            xaxis_title="pressure_axis",
            yaxis_title="pressure_rotation",
            zaxis_title=report["surface_target"],
        ),
        height=800,
    )

    surface_html = SURFACE_HTML_DIR / f"surface_{surface_type}.html"
    fig.write_html(surface_html, include_plotlyjs=True, full_html=True)
    print("Saved smoothed empirical surface:", surface_html)

surface_report_df = pd.DataFrame(surface_reports)
display(surface_report_df)
surface_report_df.to_csv(SURFACE_REPORT_PATH, index=False)
print("Saved surface report:", SURFACE_REPORT_PATH)
print("Surface target candidates:", SURFACE_TARGET_CANDIDATES)

Saved smoothed empirical surface: plotly_surfaces_html\surface_soft_low_energy.html
Saved smoothed empirical surface: plotly_surfaces_html\surface_medium_low_energy.html


Saved smoothed empirical surface: plotly_surfaces_html\surface_medium_high_energy.html
Saved smoothed empirical surface: plotly_surfaces_html\surface_hard_high_energy.html


,rock_energy_type_final,surface_target,rows_used,grid_size,min_bin_count,smooth_sigma,filled_cell_ratio_before_min_count,nonzero_bin_count,median_bin_count,pressure_axis_q_low,pressure_axis_q_high,pressure_rotation_q_low,pressure_rotation_q_high,surface_speed_min,surface_speed_median,surface_speed_max,raw_filled_cell_ratio_after_min_count,z_clip_low,z_clip_high
0,soft_low_energy,speed_roll_median_30,84601,40,5,1.6,0.780625,1249,36.0,5851.45,21365.0,7757.0,18480.0,0.008864,0.016274,0.021157,0.609375,0.008864,0.021157
1,medium_low_energy,speed_roll_median_30,83791,40,5,1.6,0.818125,1309,26.0,9369.00,22292.0,9017.0,19022.0,0.006466,0.012484,0.020981,0.637500,0.006466,0.020981
2,medium_high_energy,speed_roll_median_30,83081,40,5,1.6,0.860625,1377,27.0,12894.85,22658.0,9892.0,18892.0,0.006155,0.011766,0.013008,0.699375,0.006155,0.013008
3,hard_high_energy,speed_roll_median_30,83288,40,5,1.6,0.911875,1459,35.0,12238.00,22668.0,9400.0,18743.0,0.005555,0.006438,0.011673,0.748750,0.005555,0.011673


Saved surface report: rock_energy_segment_reports\surface_empirical_report.csv
Surface target candidates: ['speed_roll_median_30', 'speed_roll_mean_30', 'speed_roll_median_12', 'speed']


## 9. Global speed surface for simulator 3D visualization

The simulator uses one global speed response surface built from all rows. Energy quantiles remain model features/context; they are not used to split the 3D surface.


In [10]:
GLOBAL_SURFACE_PATH = SURFACE_HTML_DIR / "global_speed_surface.json"
GLOBAL_SURFACE_REPORT_PATH = SURFACE_REPORT_DIR / "global_speed_surface_report.csv"
GLOBAL_SURFACE_SOURCE = "united_rock_energy_segment_quantile.csv"


def build_global_speed_surface(
    data: pd.DataFrame,
    x_col: str = "pressure_axis",
    y_col: str = "pressure_rotation",
    z_col: str = "speed",
    grid_size: int = SURFACE_GRID_SIZE,
):
    part = data[[x_col, y_col, z_col]].copy()
    for col in [x_col, y_col, z_col]:
        part[col] = pd.to_numeric(part[col], errors="coerce")
    part = part.replace([np.inf, -np.inf], np.nan).dropna(subset=[x_col, y_col, z_col])
    part = part[part[z_col] > 0].copy()
    rows_total_after_numeric_cleaning = int(len(part))
    if rows_total_after_numeric_cleaning < 500:
        raise ValueError(f"Not enough rows to build global speed surface: {rows_total_after_numeric_cleaning}")

    q_low, q_high = SURFACE_AXIS_QUANTILES
    x_min = float(part[x_col].quantile(q_low))
    x_max = float(part[x_col].quantile(q_high))
    y_min = float(part[y_col].quantile(q_low))
    y_max = float(part[y_col].quantile(q_high))
    part = part[part[x_col].between(x_min, x_max) & part[y_col].between(y_min, y_max)].copy()
    if len(part) < 500:
        raise ValueError(f"Not enough rows inside axis quantiles to build global speed surface: {len(part)}")

    x_edges = np.linspace(x_min, x_max, grid_size + 1)
    y_edges = np.linspace(y_min, y_max, grid_size + 1)
    part["_x_bin"] = np.clip(np.digitize(part[x_col], x_edges) - 1, 0, grid_size - 1)
    part["_y_bin"] = np.clip(np.digitize(part[y_col], y_edges) - 1, 0, grid_size - 1)

    grouped = part.groupby(["_y_bin", "_x_bin"])[z_col].agg(["median", "count"]).reset_index()
    z = np.full((grid_size, grid_size), np.nan, dtype=float)
    counts = np.zeros((grid_size, grid_size), dtype=float)
    for y_idx, x_idx, z_median, z_count in grouped[["_y_bin", "_x_bin", "median", "count"]].to_numpy():
        y_idx = int(y_idx)
        x_idx = int(x_idx)
        z[y_idx, x_idx] = float(z_median)
        counts[y_idx, x_idx] = float(z_count)

    filled_ratio_before_min_count = float(np.isfinite(z).mean())
    z_smooth, smooth_report = fill_and_smooth_surface(z, counts)

    x_centers = (x_edges[:-1] + x_edges[1:]) / 2
    y_centers = (y_edges[:-1] + y_edges[1:]) / 2
    pa, pr = np.meshgrid(x_centers, y_centers)

    report = {
        "surface_name": "global_speed_surface",
        "source_data": GLOBAL_SURFACE_SOURCE,
        "x": x_col,
        "y": y_col,
        "z": z_col,
        "surface_method": "global_smoothed_empirical_binned_median_speed_surface",
        "rows_total_after_numeric_cleaning": rows_total_after_numeric_cleaning,
        "rows_used_inside_axis_quantiles": int(len(part)),
        "grid_size": grid_size,
        "min_bin_count": SURFACE_MIN_BIN_COUNT,
        "smooth_sigma": SURFACE_SMOOTH_SIGMA,
        "axis_quantile_low": q_low,
        "axis_quantile_high": q_high,
        "x_min": x_min,
        "x_max": x_max,
        "y_min": y_min,
        "y_max": y_max,
        "filled_cell_ratio_before_min_count": filled_ratio_before_min_count,
        **smooth_report,
        "speed_min": float(np.nanmin(z_smooth)),
        "speed_median": float(np.nanmedian(z_smooth)),
        "speed_max": float(np.nanmax(z_smooth)),
    }
    return pa, pr, z_smooth, report


global_pa, global_pr, global_z, global_surface_report = build_global_speed_surface(df_out)
GLOBAL_SURFACE_PATH.write_text(
    json.dumps(
        {
            "source": "global_speed_surface",
            "x": global_pa.tolist(),
            "y": global_pr.tolist(),
            "z": global_z.tolist(),
            "metadata": global_surface_report,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
pd.DataFrame([global_surface_report]).to_csv(GLOBAL_SURFACE_REPORT_PATH, index=False)

print("Saved global speed surface:", GLOBAL_SURFACE_PATH)
print("Saved global speed surface report:", GLOBAL_SURFACE_REPORT_PATH)
display(pd.DataFrame([global_surface_report]))


Saved global speed surface: plotly_surfaces_html\global_speed_surface.json
Saved global speed surface report: rock_energy_segment_reports\global_speed_surface_report.csv


,surface_name,source_data,x,y,z,surface_method,rows_total_after_numeric_cleaning,rows_used_inside_axis_quantiles,grid_size,min_bin_count,smooth_sigma,axis_quantile_low,axis_quantile_high,x_min,x_max,y_min,y_max,filled_cell_ratio_before_min_count,raw_filled_cell_ratio_after_min_count,z_clip_low,z_clip_high,speed_min,speed_median,speed_max
0,global_speed_surface,united_rock_energy_segment_quantile.csv,pressure_axis,pressure_rotation,speed,global_smoothed_empirical_binned_median_speed_...,415049,348705,40,5,1.6,0.05,0.95,6645.0,22343.0,8246.0,18758.6,0.92875,0.7725,0.004794,0.018627,0.004794,0.012663,0.018627


In [11]:
OUTPUT_PATH = "united_rock_energy_segment_quantile.csv"
CONFIG_PATH = "rock_energy_segment_quantile_config.json"

if "depth_m" not in df_out.columns:
    if "depth" not in df_out.columns:
        raise ValueError("Missing depth/depth_m in final df_out")
    df_out["depth_m"] = df_out["depth"]
df_out["depth_m"] = pd.to_numeric(df_out["depth_m"], errors="coerce")

front_cols = [
    "processing_time",
    "depth_m",
    "rotation",
    "pressure_axis",
    "pressure_rotation",
    "well_id",
    "speed",
]
remaining_cols = [col for col in df_out.columns if col not in front_cols and col != "depth"]
df_out = df_out[front_cols + remaining_cols]

df_out.to_csv(OUTPUT_PATH, index=False)

config = {
    "data_path": DATA_PATH,
    "output_path": OUTPUT_PATH,
    "final_method": "energy_type_segment_quantile_log_pseudo_mse_only",
    "segment_size": SEGMENT_SIZE,
    "labels": energy_labels_4,
    "hardness_features": HARDNESS_FEATURES,
    "hardness_base_signal": "log_pseudo_mse_roll_median_30",
    "hardness_formula": "zscore(log_pseudo_mse_roll_median_30)",
    "energy_quantile_source": "hardness_score_smooth segment median",
    "surface_method": "smoothed_empirical_binned_median_surface",
    "surface_target": "per-energy first available candidate",
    "surface_target_candidates": SURFACE_TARGET_CANDIDATES,
    "surface_grid_size": SURFACE_GRID_SIZE,
    "surface_min_bin_count": SURFACE_MIN_BIN_COUNT,
    "surface_smooth_sigma": SURFACE_SMOOTH_SIGMA,
    "surface_z_clip_quantiles": list(SURFACE_Z_CLIP_QUANTILES),
    "surface_axis_quantiles": list(SURFACE_AXIS_QUANTILES),
    "surface_interpolation": "pandas linear interpolation followed by weighted gaussian smoothing over binned median grid",
    "surface_report_path": str(SURFACE_REPORT_PATH),
    "simulator_visual_surface_method": "global_smoothed_empirical_binned_median_speed_surface",
    "simulator_visual_surface_path": str(GLOBAL_SURFACE_PATH),
    "simulator_visual_surface_report_path": str(GLOBAL_SURFACE_REPORT_PATH),
    "depth_column": "depth_m",
    "interpretation": {
        "hardness_score_smooth": "stable operational drilling resistance index based on smoothed log-pseudo-MSE",
        "rock_energy_type_final": "final discrete energy-response regime from segment-level quantile segmentation",
        "pseudo_mse": "proxy energy per penetration, not physical MSE",
        "plotly_surfaces": "legacy per-energy smoothed empirical surfaces; simulator 3D now uses a single global speed surface",
        "simulator_visual_surface": "single global speed response surface; energy classes remain model features and context",
    },
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_PATH)
print("Saved:", CONFIG_PATH)
print("Output has depth_m:", "depth_m" in df_out.columns)
print("Output columns head:", list(df_out.columns[:10]))


Saved: united_rock_energy_segment_quantile.csv
Saved: rock_energy_segment_quantile_config.json
Output has depth_m: True
Output columns head: ['processing_time', 'depth_m', 'rotation', 'pressure_axis', 'pressure_rotation', 'well_id', 'speed', 'dt', 'total_pressure', 'pressure_balance']
